# Project 07 — 05 — Deflated Sharpe Ratio


## Notebook introduction

- **Concepts required:** Sharpe ratio, sample skewness/kurtosis, expected maximum under a null.
- **What will be learned:** How to deflate a Sharpe by the number of trials and non-normality (PSR, Deflated Sharpe, minimum track-record length).
- **Expected mathematical / statistical background:** Moments of a distribution and the normal CDF.
- **Outputs produced:** PSR, Deflated Sharpe probability, and minTRL; `results/deflated_sharpe.csv`; evidence records.
- **How this notebook feeds later notebooks:** Provides the selection-aware probability that becomes a primary gate in the report (06).


## Purpose

Convert the raw, search-inflated Sharpe into a selection-aware probability: compute the Probabilistic and Deflated Sharpe Ratio and the minimum track-record length, penalizing for trials and non-normality.

> **Scaffold notebook.** Section headings only — no implementation yet. Algorithms live in `src/statistics/deflated_sharpe.py` (currently raising `NotImplementedError`).


## Expected inputs

- Reference strategy returns from the frozen Project 06 baseline (`experiments/completed/exp_005_risk_engine_final/final_returns_v2_with_dates.csv`)
- Run configuration (seed, iteration counts, significance level)
- Number of trials in the search family and the cross-trial Sharpe variance


## Expected outputs

- Probabilistic Sharpe Ratio (vs benchmark 0)
- Deflated Sharpe Ratio and the expected-max-Sharpe benchmark
- Minimum track-record length
- `results/deflated_sharpe.csv`
- Evidence-ledger records


## Setup & inputs


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.metrics import sharpe_ratio                       # annualized (reference only)
from src.statistics.deflated_sharpe import (                     # reused; no formulas duplicated
    probabilistic_sharpe_ratio,
    expected_max_sharpe,
    deflated_sharpe_ratio,
    minimum_track_record_length,
)

# Same Project 07 candidate series used for PSR/DSR (frozen Project 06 baseline).
EXP_DIR = PROJECT_ROOT / "experiments/completed/exp_005_risk_engine_final"
RETURNS_FILE = EXP_DIR / "final_weighted_multi_strategy_portfolio_dd.csv"
WEIGHTS_FILE = EXP_DIR / "final_daily_weights.csv"
STRATEGY_NAME = ("Project 05 final weighted multi-strategy portfolio (DD-overlay) "
                 "— frozen Project 06 baseline")

ret_df = pd.read_csv(RETURNS_FILE)
dates = (pd.read_csv(WEIGHTS_FILE, parse_dates=["Date"])
         .pivot(index="Date", columns="Ticker", values="Weight").sort_index().index)
assert len(dates) == len(ret_df)
returns = pd.Series(ret_df["portfolio_return"].to_numpy(), index=dates, name="returns")

PERIODS_PER_YEAR = 252
print("candidate :", STRATEGY_NAME)
print("obs T     :", len(returns), "| range:", returns.index.min().date(), "->", returns.index.max().date())

candidate : Project 05 final weighted multi-strategy portfolio (DD-overlay) — frozen Project 06 baseline
obs T     : 2558 | range: 2016-01-04 -> 2026-03-06


## Sample moments (Sharpe, skew, kurtosis, n)


In [2]:
# 1. Sample moments (all Sharpe on the per-period scale; annualized for reference).
T = int(returns.shape[0])
mean = float(returns.mean())
std = float(returns.std(ddof=1))
sr_pp = mean / std                                   # per-period Sharpe
sr_ann = sharpe_ratio(returns)                       # annualized (x sqrt(252))
skew = float(returns.skew())
kurt_pearson = float(returns.kurt()) + 3.0           # ordinary/Pearson (Normal = 3)

moments = pd.Series({
    "T": T, "mean": mean, "std (ddof=1)": std,
    "per-period Sharpe": sr_pp, "annualized Sharpe": sr_ann,
    "skewness": skew, "Pearson kurtosis": kurt_pearson,
})
moments.to_frame("candidate")

,candidate
T,2558.000000
mean,0.001320
std (ddof=1),0.010027
per-period Sharpe,0.131665
annualized Sharpe,2.090113
skewness,0.996546
Pearson kurtosis,9.717704


## Probabilistic Sharpe Ratio


`PSR(SR*=0)` tests whether the strategy's **true Sharpe exceeds zero**, after accounting for the finite sample size *T*, the return **skewness**, and the **kurtosis** (non-normality). It is a probability, not a p-value: values near 1 indicate strong evidence the true Sharpe is positive.

In [3]:
psr0 = probabilistic_sharpe_ratio(returns, benchmark_sharpe=0.0)
print(f"PSR(SR*=0) = {psr0:.12f}")

PSR(SR*=0) = 0.999999999999


## Expected maximum Sharpe under the null


### Trial universe for the multiple-testing benchmark

The Deflated Sharpe benchmark needs the **variance of trial Sharpes** on the per-period scale. We use the **23 series-backed trials** that plausibly informed selecting this candidate: 12 Project-04 base strategies (`exp_004…/strategy_timeseries`) + 11 Project-05/06 US risk-overlay variants (`overlay_equity_curves`). Sharpes are recomputed from the actual return series.

**High-dependence caveat.** These variants are near-duplicates of one US long-short alpha: mean pairwise return correlation ≈ 0.94 and a participation-ratio effective dimension `N_eff = (trC)²/tr(C²) ≈ 1.1`. So this variance (and the raw trial count) reflect a highly correlated search — the effective number of *independent* trials is small and genuinely uncertain.

In [4]:
import glob, os

# Family A: Project 04 base strategies (portfolio_return series).
A = {}
for f in sorted(glob.glob(str(PROJECT_ROOT / "experiments/completed/exp_004_project04_final/strategy_timeseries/*.csv"))):
    A[os.path.basename(f)[:-4]] = pd.read_csv(f)["portfolio_return"].to_numpy()
A = pd.DataFrame(A)

# Family B: US risk-overlay variants (returns from equity curves).
eqB = pd.read_csv(PROJECT_ROOT / "experiments/completed/exp_006_deployment_overlay_challenge/overlay_equity_curves.csv").drop(columns=["Date"])
B = eqB.pct_change().dropna().reset_index(drop=True)

per_period = lambda s: s.mean() / s.std(ddof=1)
trial_sharpes = pd.concat([A.apply(per_period), B.apply(per_period)])   # 23 per-period Sharpes
n_trials_backed = int(trial_sharpes.shape[0])
V_trials = float(trial_sharpes.var(ddof=1))
std_trials = float(trial_sharpes.std(ddof=1))

# Dependence diagnostics (positional alignment over the shared daily calendar).
comb = pd.concat([A.iloc[1:].reset_index(drop=True).add_prefix("A:"), B.add_prefix("B:")], axis=1)
C = comb.corr().values
mean_offdiag = float(C[np.triu_indices_from(C, k=1)].mean())
w = np.clip(np.linalg.eigvalsh(C), 0, None)
N_eff = float((w.sum() ** 2) / np.sum(w ** 2))

print(f"series-backed trials       : {n_trials_backed} (12 base + 11 overlay)")
print(f"V[per-period Sharpe]       : {V_trials:.3e}  (std {std_trials:.5f})   [= 4.920e-4]")
print(f"mean pairwise correlation  : {mean_offdiag:.4f}")
print(f"participation-ratio N_eff  : {N_eff:.2f}  of {n_trials_backed}")

series-backed trials       : 23 (12 base + 11 overlay)
V[per-period Sharpe]       : 4.920e-04  (std 0.02218)   [= 4.920e-4]
mean pairwise correlation  : 0.9389
participation-ratio N_eff  : 1.13  of 23


### Expected-maximum-Sharpe hurdle across trial-count scenarios

`SR*` is the selection-inflated benchmark (expected maximum Sharpe under the null across *N* independent trials). Because the effective *N* is genuinely ambiguous, we show three defensible scenarios rather than one canonical value.

In [5]:
SCENARIOS = {
    2:   "correlation-adjusted (N_eff ~ 1.1, floored to 2)",
    5:   "distinct-family",
    100: "conservative / high-search",
}

rows = []
for N in SCENARIOS:
    sr_star_pp = expected_max_sharpe(n_trials=N, trial_sharpe_variance=V_trials)
    rows.append({"N": N, "scenario": SCENARIOS[N],
                 "SR* per-period": sr_star_pp,
                 "SR* annualized": sr_star_pp * np.sqrt(PERIODS_PER_YEAR)})
sr_star_table = pd.DataFrame(rows).set_index("N")
sr_star_table.round(6)

,scenario,SR* per-period,SR* annualized
N,,,
2,"correlation-adjusted (N_eff ~ 1.1, floored to 2)",0.011529,0.183019
5,distinct-family,0.026454,0.419942
100,conservative / high-search,0.056133,0.891089


## Deflated Sharpe Ratio


In [6]:
# 5. DSR sensitivity table (DSR = PSR at the selection-inflated benchmark; per-period internally).
rows = []
for N in SCENARIOS:
    res = deflated_sharpe_ratio(returns, n_trials=N, trial_sharpe_variance=V_trials)
    rows.append({
        "N": N,
        "SR* per-period": res.expected_max_sharpe,
        "SR* annualized": res.expected_max_sharpe * np.sqrt(PERIODS_PER_YEAR),
        "observed SR per-period": res.observed_sharpe,
        "observed SR annualized": res.observed_sharpe * np.sqrt(PERIODS_PER_YEAR),
        "DSR": res.deflated_probability,
    })
dsr_table = pd.DataFrame(rows).set_index("N")
dsr_table.round(12)

,SR* per-period,SR* annualized,observed SR per-period,observed SR annualized,DSR
N,,,,,
2,0.011529,0.183019,0.131665,2.090113,1.00000
5,0.026454,0.419942,0.131665,2.090113,1.00000
100,0.056133,0.891089,0.131665,2.090113,0.99997


### Interpretation

- **DSR remains very high across all three scenarios** (≥ ~0.99997 for *N* from 2 to 100): the observed per-period Sharpe (~0.132; ~2.09 annualized) stays far above the selection-inflated hurdle `SR*` (annualized ~0.18 → ~0.89) in every case.
- **This conclusion is conditional** on the assumed number of effective trials *N* and on the representative trial-Sharpe variance `V ≈ 4.92e-4`. Both are modelling choices, not facts.
- The **remaining uncertainty lies in reconstructing the effective research search space** (how many *independent* trials, and how dispersed a genuinely independent search would be), **not in the DSR formula itself**. Note the variance here comes from a near-duplicate universe (`N_eff ≈ 1.1`), which may *understate* the dispersion of a truly independent search and hence make DSR optimistic. No scenario is the definitive DSR, and this is not a deployment verdict.

## Minimum track-record length


`minimum_track_record_length()` inverts PSR for the sample length *T*: the observations required for the observed Sharpe to be significant above a benchmark at a given confidence. Raw *T_min* is real-valued; we `ceil` it to whole observations and convert to years at 252 trading days. Two benchmarks: `SR*=0` (Sharpe > 0) and the conservative DSR hurdle (`SR*_pp ≈ 0.05613`, annualized ≈ 0.89).

In [7]:
import math

mintrl_cases = [
    (0.0, 0.95), (0.0, 0.99),
    (0.05613, 0.95), (0.05613, 0.99),   # conservative DSR hurdle (per-period)
]
rows = []
for bench, c in mintrl_cases:
    tmin = minimum_track_record_length(returns, benchmark_sharpe=bench, confidence_level=c)
    rows.append({
        "SR* per-period": bench,
        "SR* annualized": bench * np.sqrt(PERIODS_PER_YEAR),
        "confidence": c,
        "T_min (raw)": tmin,
        "T_min (ceil obs)": math.ceil(tmin),
        "approx years": tmin / PERIODS_PER_YEAR,
        "actual T": T,
        "actual / required": T / tmin,
    })
mintrl_table = pd.DataFrame(rows)
mintrl_table.round(4)

,SR* per-period,SR* annualized,confidence,T_min (raw),T_min (ceil obs),approx years,actual T,actual / required
0,0.0000,0.000,0.95,142.4874,143,0.5654,2558,17.9525
1,0.0000,0.000,0.99,284.0171,285,1.1271,2558,9.0065
2,0.0561,0.891,0.95,430.8960,431,1.7099,2558,5.9365
3,0.0561,0.891,0.99,860.9204,861,3.4164,2558,2.9712


**Reading.** Proving the Sharpe is above **zero** needs only ~143 obs (~0.6 yr) at 95% — the candidate's T=2558 is ~18× that. Against the **conservative DSR hurdle** at 99% it needs ~861 obs (~3.4 yr), still comfortably within the available history (~3×). *T_min* roughly doubles from 0.95→0.99 (the Φ⁻¹(c)² factor) and grows sharply as the benchmark approaches the observed Sharpe (the 1/(SR̂−SR*)² blow-up).

## Results table


In [8]:
# Consolidated results for the record.
print(f"PSR(SR*=0) = {psr0:.12f}")
dsr_table.round(12)

PSR(SR*=0) = 0.999999999999


,SR* per-period,SR* annualized,observed SR per-period,observed SR annualized,DSR
N,,,,,
2,0.011529,0.183019,0.131665,2.090113,1.00000
5,0.026454,0.419942,0.131665,2.090113,1.00000
100,0.056133,0.891089,0.131665,2.090113,0.99997


## Evidence ledger

Record every conclusion produced above as an `EvidenceRecord` (`src/statistics/evidence.py`): strategy version, Project 06 freeze tag, dataset, date range, parameters, seed, method, assumptions, statistic, p-value / interval, decision, limitations, timestamp.


In [9]:
from datetime import datetime, timezone
from src.statistics.evidence import EvidenceRecord, EvidenceLedger, Decision, StatisticalMethod

scenario_records = [
    {"N": int(N),
     "sr_star_per_period": float(deflated_sharpe_ratio(returns, n_trials=N, trial_sharpe_variance=V_trials).expected_max_sharpe),
     "sr_star_annualized": float(deflated_sharpe_ratio(returns, n_trials=N, trial_sharpe_variance=V_trials).expected_max_sharpe * np.sqrt(PERIODS_PER_YEAR)),
     "dsr": float(deflated_sharpe_ratio(returns, n_trials=N, trial_sharpe_variance=V_trials).deflated_probability)}
    for N in SCENARIOS
]

record = EvidenceRecord(
    record_id="p07_nb05_deflated_sharpe_v1",
    strategy_version=STRATEGY_NAME,
    project06_freeze_tag="project-06-deployment-validation-frozen",
    experiment_id="exp_005_risk_engine_final",
    dataset=str(RETURNS_FILE.relative_to(PROJECT_ROOT)),
    date_range=(str(returns.index.min().date()), str(returns.index.max().date())),
    parameters={
        "scale": "per-period (annualized shown for reference only)",
        "psr_benchmark": 0.0,
        "trial_sharpe_variance": V_trials,
        "n_trials_scenarios": list(SCENARIOS.keys()),
        "scenario_labels": SCENARIOS,
        "trial_universe": "23 series-backed (12 P04 base + 11 US overlay)",
    },
    random_seed=None,   # PSR/DSR are deterministic closed-form (no RNG)
    notebook="research/project_07_statistical_integrity/notebooks/05_deflated_sharpe.ipynb",
    source_module="src/statistics/deflated_sharpe.py",
    method=StatisticalMethod.DEFLATED_SHARPE,
    method_version="stat_v1",
    assumptions=(
        "PSR/DSR (Bailey-Lopez de Prado, AFML) on the per-period Sharpe; non-normality via sample "
        "skew and Pearson kurtosis; DSR benchmark = expected max Sharpe across N independent trials; "
        "trial-Sharpe variance estimated from 23 near-duplicate series (participation-ratio "
        "N_eff ~ 1.1), so effective N and representative variance are modelling choices."
    ),
    hypothesis=(
        "Does the candidate Sharpe remain significant after finite-sample non-normality (PSR) and "
        "after multiple-testing selection deflation (DSR) across a range of assumed effective N?"
    ),
    test_statistic=float(sr_pp),                       # observed per-period Sharpe
    confidence_level=None,
    deflated_sharpe_result=float(scenario_records[-1]["dsr"]),   # conservative N=100 scenario
    n_trials=None,                                     # effective N deliberately NOT fixed to one value
    decision=Decision.INCONCLUSIVE,                    # DSR high, but conditional on N and V
    limitations=(
        "DSR >= ~0.99997 across N in {2,5,100} GIVEN V=4.92e-4, but V comes from a near-duplicate "
        "trial universe (may understate a truly-independent search's dispersion, making DSR "
        "optimistic); effective number of independent trials is uncertain; PSR/DSR do not establish "
        "deployable alpha and are not a forecast."
    ),
    timestamp=datetime.now(timezone.utc),
    extra={
        "moments": {"T": T, "mean": mean, "std": std, "per_period_sharpe": sr_pp,
                    "annualized_sharpe": float(sr_ann), "skew": skew, "pearson_kurtosis": kurt_pearson},
        "psr_sr0": float(psr0),
        "trial_universe": {"n_series_backed": n_trials_backed, "V_per_period": V_trials,
                           "std_per_period": std_trials, "mean_pairwise_corr": mean_offdiag,
                           "participation_ratio_N_eff": N_eff},
        "dsr_scenarios": scenario_records,
        "status": "PSR/DSR evidence completed",
    },
)

ledger = EvidenceLedger()
try:
    ledger.append(record)
    print("Record appended to the Evidence Ledger.")
except NotImplementedError:
    print("Evidence Ledger persistence not implemented yet (milestone P07-6);")
    print("record constructed against the canonical schema and displayed below (not persisted).")

Evidence Ledger persistence not implemented yet (milestone P07-6);
record constructed against the canonical schema and displayed below (not persisted).


In [10]:
import dataclasses, pprint
_rec = dataclasses.asdict(record)
_rec["method"] = record.method.value
_rec["decision"] = record.decision.value
_rec["timestamp"] = record.timestamp.isoformat()
pprint.pp(_rec, sort_dicts=False, width=100)

{'record_id': 'p07_nb05_deflated_sharpe_v1',
 'strategy_version': 'Project 05 final weighted multi-strategy portfolio (DD-overlay) — frozen '
                     'Project 06 baseline',
 'project06_freeze_tag': 'project-06-deployment-validation-frozen',
 'experiment_id': 'exp_005_risk_engine_final',
 'dataset': 'experiments/completed/exp_005_risk_engine_final/final_weighted_multi_strategy_portfolio_dd.csv',
 'date_range': ('2016-01-04', '2026-03-06'),
 'parameters': {'scale': 'per-period (annualized shown for reference only)',
                'psr_benchmark': 0.0,
                'trial_sharpe_variance': 0.000492032449377448,
                'n_trials_scenarios': [2, 5, 100],
                'scenario_labels': {2: 'correlation-adjusted (N_eff ~ 1.1, floored to 2)',
                                    5: 'distinct-family',
                                    100: 'conservative / high-search'},
                'trial_universe': '23 series-backed (12 P04 base + 11 US overlay)'},
 'random_

## Conclusions


## Conclusions & status

- **Candidate:** per-period Sharpe ≈ 0.1316 (annualized ≈ 2.09), T = 2558, positive skew (~1.0), heavy Pearson kurtosis (~9.7).
- **PSR(SR\*=0) ≈ 1:** the true Sharpe is almost surely positive after finite-sample non-normality.
- **DSR ≈ 1 across N ∈ {2, 5, 100}** (given V ≈ 4.92e-4): the Sharpe survives multiple-testing deflation for every tested effective-trial count — *conditional* on the assumed N and variance.
- **Open uncertainty** is reconstructing the effective search space (independent N and a representative independent-search variance), not the DSR formula. No scenario is definitive; this is not a deployment verdict.

- **Minimum track-record length:** ~143 obs (~0.6 yr) to prove Sharpe > 0 at 95%; ~861 obs (~3.4 yr) against the conservative DSR hurdle at 99% — both within the available T=2558.

**Notebook 05 (Deflated Sharpe Ratio) — COMPLETE.** PSR, expected-max Sharpe, DSR sensitivity, and minimum track-record length are all included.